In [ ]:
# version1_bow_naive_bayes.py
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from datasets import load_dataset
import matplotlib.pyplot as plt
import seaborn as sns
import time

print("Loading IMDB dataset...")
dataset = load_dataset("imdb")

# Prepare data
X_train = dataset['train']['text']
y_train = dataset['train']['label']
X_test = dataset['test']['text']
y_test = dataset['test']['label']

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# Method 1: Bag of Words (Count Vectorizer)
print("\n=== Bag of Words + Naive Bayes ===")
start_time = time.time()

vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

# Train Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

train_time = time.time() - start_time

# Predictions
start_inference = time.time()
y_pred = nb_model.predict(X_test_bow)
inference_time = time.time() - start_inference

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTraining Time: {train_time:.2f}s")
print(f"Inference Time: {inference_time:.4f}s")
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")

print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix - Bag of Words + Naive Bayes')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('bow_confusion_matrix.png')
print("Confusion matrix saved as 'bow_confusion_matrix.png'")

# Test on sample reviews
sample_reviews = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "Terrible film, waste of time and money. Very disappointed.",
    "It was okay, nothing special but not bad either."
]


print("\n=== Sample Predictions ===")
for review in sample_reviews:
    prediction = nb_model.predict(vectorizer.transform([review]))[0]
    sentiment = "Positive" if prediction == 1 else "Negative"
    print(f"Review: {review[:50]}...")
    print(f"Prediction: {sentiment}\n")